<a href="https://colab.research.google.com/github/m-zayed5722/Miscellaneous-Projects/blob/main/AgentRouter_Lite.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AgentRouter Lite: Route Queries to the Right Tool (Colab)

Pipeline:
User Query
→ Intent Detection
→ Tool Selection
→ Tool Execution
→ Final Response + Routing Trace

Tools:
- SQL tool
- RAG tool
- Calculator tool
- Generic text tool

Goals:
- Show agentic orchestration
- Keep routing explainable
- Run fully in Colab with no API key

In [3]:
#!pip -q install duckdb pandas

In [2]:
import re
import math
import duckdb
import pandas as pd
from dataclasses import dataclass
from typing import Dict, Any, Optional, List

In [4]:
data = [
    {"app_name": "Atlas",   "team": "Platform", "director": "Patel",  "cost_usd_per_month": 12000, "incidents_last_30d": 2, "monthly_active_users": 40000},
    {"app_name": "Nimbus",  "team": "Payments", "director": "Chen",   "cost_usd_per_month": 22000, "incidents_last_30d": 5, "monthly_active_users": 70000},
    {"app_name": "Orion",   "team": "Core",     "director": "Singh",  "cost_usd_per_month": 18000, "incidents_last_30d": 1, "monthly_active_users": 55000},
    {"app_name": "Pulse",   "team": "Mobile",   "director": "Garcia", "cost_usd_per_month": 14000, "incidents_last_30d": 4, "monthly_active_users": 62000},
    {"app_name": "Beacon",  "team": "Data",     "director": "Nguyen", "cost_usd_per_month": 26000, "incidents_last_30d": 3, "monthly_active_users": 81000},
]

df_apps = pd.DataFrame(data)

con = duckdb.connect(database=":memory:")
con.execute("CREATE TABLE apps AS SELECT * FROM df_apps")
con.execute("SELECT * FROM apps").fetchdf()

,app_name,team,director,cost_usd_per_month,incidents_last_30d,monthly_active_users
0,Atlas,Platform,Patel,12000,2,40000
1,Nimbus,Payments,Chen,22000,5,70000
2,Orion,Core,Singh,18000,1,55000
3,Pulse,Mobile,Garcia,14000,4,62000
4,Beacon,Data,Nguyen,26000,3,81000


In [5]:
DOCS = [
    {
        "doc_id": "doc_1",
        "title": "RAG Design Notes",
        "text": """
        Retrieval-Augmented Generation (RAG) combines document retrieval with answer generation.
        The retriever finds relevant chunks from a knowledge base, and the generator uses those chunks
        to answer questions more faithfully. Citations improve trust and make answers easier to verify.
        """
    },
    {
        "doc_id": "doc_2",
        "title": "Text-to-SQL Guardrails",
        "text": """
        Text-to-SQL systems should use read-only queries, table allowlists, column allowlists, query limits,
        and timeouts. They should also log generated SQL and reject dangerous statements like DROP, DELETE, or UPDATE.
        """
    },
    {
        "doc_id": "doc_3",
        "title": "LLM Evaluation Basics",
        "text": """
        Evaluation frameworks for LLM systems often score correctness, completeness, clarity, and faithfulness.
        Pairwise comparisons are useful when direct scoring is noisy.
        """
    }
]